In [ ]:
import tensorflow as tf
import tensorflow_io as tfio
import os
import pandas as pd
from IPython import display

In [ ]:
class Config:
    competition = "birdclef-2023"
    
    frame_duration = 5 # seconde
    sample_rate = 32_000
    desired_sample_rate = 16_000
    frame_length = frame_duration*desired_sample_rate
    output = f"{competition}/"
    train_dir = os.path.join(output, "train")
    valid_dir = os.path.join(output, "validation")
    
    root = "/kaggle/input/birdclef-2023"
    train_dir_inp = os.path.join(root, "train_audio/")
    metada_file = os.path.join(root, "train_metadata.csv")
    
    records_size = 2024

In [ ]:
# Check GPU
if tf.config.list_physical_devices('GPU'):
    strategy = tf.distribute.MirroredStrategy()
    device = "GPU"
# Use CPU
else:
    strategy = tf.distribute.get_strategy()
    device = "CPU"

print("Number of accelerators ", strategy.num_replicas_in_sync,"|", device)

In [ ]:
def audio2frames(audio, label):
    #----------------------
    # Audio
    #----------------------
    frames = tf.signal.frame(
        audio, 
        frame_length=Config.frame_length, 
        frame_step=Config.frame_length,
        pad_end=True, 
        name="frames_audio"
    )
    #----------------------
    # Label
    #----------------------
    nlab = tf.shape(frames)[0]
    label = tf.repeat(label, repeats=nlab)
    return frames, label

In [ ]:
def read_file(path, label):
    file = tf.io.read_file(path)
    aud = tfio.audio.decode_vorbis(file)
    aud = tfio.audio.resample(aud, rate_in=Config.sample_rate, rate_out=Config.desired_sample_rate)
    aud = tf.squeeze(aud, axis=-1)
    aud = tf.cast(aud, tf.float32)
    frames, label = audio2frames(aud, label)
    return frames, label 

In [ ]:
df = pd.read_csv(Config.metada_file)
encode = dict(map(reversed, dict(enumerate(df["primary_label"].unique(), 1)).items()))
df["filepath"] = df["filename"].map(lambda x: os.path.join(Config.train_dir_inp, x))
df["label"] = df["primary_label"].replace(encode)
df.head()

In [ ]:
sample = df.sample(1)
path = sample["filepath"].values[0]
label = sample["label"].values[0]
frame, label = read_file(path, label)
display.Audio(frame[0].numpy(), rate=Config.desired_sample_rate)

In [ ]:
def standardize_tensor(X, feature_range=(0,1), copy=True, clip=False):
    X_min = tf.reduce_min(X, axis=0)
    X_max = tf.reduce_max(X, axis=0)
    X_std = (X - X_min) / (X_max - X_min)
    X_scaled = X_std * (feature_range[1] - feature_range[0]) + feature_range[0]
    if clip:
        X_scaled = tf.clip_by_value(X_scaled, feature_range[0], feature_range[1])
    if copy:
        X_scaled = tf.identity(X_scaled)
    return X_scaled


In [ ]:
from tqdm.notebook import tqdm
import tensorflow_hub as hub

yamnetlayer = hub.KerasLayer('https://tfhub.dev/google/yamnet/1', trainable=False)



def extract_log_mel(frame):
    frame = standardize_tensor(frame)
    _, _, log_mel = yamnetlayer(frame)
    return log_mel

def create_dataset(paths, labels):
    with strategy.scope():
        dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
        dataset = dataset.map(read_file, num_parallel_calls=tf.data.AUTOTUNE)
        dataset = dataset.prefetch(buffer_size=tf.data.AUTOTUNE)
    return dataset

In [ ]:
lg_mel = extract_log_mel(standardize_tensor(frame[0], feature_range=[-1,1]))

In [ ]:
def _bytes_feature(value):
    """Returns a bytes_list from a string / byte."""
    if isinstance(value, type(tf.constant(0))): # if value ist tensor
        value = value.numpy() # get value of tensor
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))

def _int64_feature(value):
    """Returns an int64_list from a bool / enum / int / uint."""
    return tf.train.Feature(int64_list=tf.train.Int64List(value=[value]))

def serialize_array(array):
    array = tf.io.serialize_tensor(array)
    return array

def create_example(frame, label):
    log_mel = extract_log_mel(standardize_tensor(frame, feature_range=[-1, 1]))
    feature = {
        "log_mel": _bytes_feature(serialize_array(log_mel)),
        "label": _int64_feature(label)
    }
    example = tf.train.Example(features=tf.train.Features(feature=feature))
    return example

In [ ]:
def frame2records(paths, labels, path_records, record_size=2048, name='train'):
    num_records = 0
    count = 0
    writer = None
    dataset = create_dataset(paths, labels)
    total = 0 
    os.makedirs(path_records, exist_ok=True)
    for fl, ll in tqdm(dataset):
        ll = tf.cast(ll, tf.int16)
        examples = map(create_example, tf.unstack(fl, axis=0), tf.unstack(ll, axis=0))
        for example in examples:
            if writer is None:
                file = f"{path_records}/file_{num_records}.tfrec"
                writer = tf.io.TFRecordWriter(file)
            writer.write(example.SerializeToString())
            count += 1
            total += 1
            if count > record_size:
                writer.close()
                num_records += 1
                writer = None
                count = 0
                
    if writer is not None:
        writer.close()
        num_records += 1
        
    print("| Write a metadata file --------------------------")
    dff = pd.DataFrame([[len(paths), record_size, total, Config.frame_duration, Config.desired_sample_rate]], 
                   columns=["Shape", "Records_size", "NumberOfRecords", "frame_duration", "sample_rate"])
    dff.to_csv(os.path.join(Config.output, f"{name}.csv"))
    print("---------------------> Metadata saved with success")
    return None

In [ ]:
# Test 
sample = df.sample(10)
filepaths = sample["filepath"]
labels = sample["label"]
directory = os.path.join(Config.output, "test")
frame2records(filepaths, labels, path_records=directory, name='test', record_size=512)

In [ ]:
!rm -r $Config.output

In [ ]:
from sklearn.model_selection import train_test_split
def train_split_label_balanced(df, labels, test_size=0.3, random_state=1024, seuil=8):
    # Like we observe before some class are less represented in the dataset 
    # So We want to define this function to split the data taking into account 
    # The weigth of the labels and we define a int seuil for the class that only get
    # Get in the train_df, in this way the train set will complete will all label sample
    df_train, df_valid = None, None 
    for lab in tqdm(labels):
        dff = df[df["primary_label"] == lab]
        if dff.shape[0] <= seuil:
            df_train = dff.copy() if df_train is None else pd.concat([df_train, dff.copy()])
        else:
            dff_train, dff_valid = train_test_split(dff, test_size=test_size, random_state=random_state)
            df_train = dff_train if df_train is None else pd.concat([df_train, dff_train])
            df_valid = dff_valid if df_valid is None else pd.concat([df_valid, dff_valid])
    return df_train, df_valid

In [ ]:
from sklearn.utils import shuffle
df = shuffle(df, random_state=1024)

train_df, valid_df = train_split_label_balanced(df, labels=df["primary_label"].unique(), test_size=0.3)

In [ ]:
Config.train_dir

In [ ]:
print("Training size | Size: ", len(train_df))
filepaths = train_df["filepath"]
labels = train_df["label"]
frame2records(filepaths, labels, path_records=Config.train_dir, record_size=Config.records_size, name="train")

In [ ]:
Config.valid_dir

In [ ]:
print("Validation size | Size: ", len(valid_df))
filepaths = valid_df["filepath"]
labels = valid_df["label"]
frame2records(filepaths, labels, path_records=Config.valid_dir, record_size=Config.records_size, name="validation")

In [ ]:
df_train_metadata = pd.read_csv(os.path.join("/kaggle/working/birdclef-2023", "train.csv"))
df_train_metadata

In [ ]:
df_train_metadata = pd.read_csv(os.path.join("/kaggle/working/birdclef-2023", "validation.csv"))
df_train_metadata

In [ ]:
print("Nice Notebook !!! -> End")